# 02 · Limitation analysis on a trained run (CPU runtime is enough)
Inputs: `dump_model_best/dump_*.pt` from notebook 01 and the frame cache (only for thumbnails).

In [ ]:
# ==== 0. Settings (edit if needed) ====
DRIVE_DIR  = "/content/drive/MyDrive/컴비 프로젝트"  # Google Drive folder (persistent: data copy, checkpoints, runs)
DRIVE_ROOT = "/content/cv_drive"                   # space-free alias (symlink) of DRIVE_DIR, used in all shell commands
REPO_URL   = "https://github.com/beblack127/computervision.git"
BRANCH     = "main"
REPO       = "/content/computervision"
LOCAL_DATA = "/content/data/msrvtt"                # fast local disk (wiped when runtime ends)
CACHE_DIR  = "/content/cache/msrvtt_u12"           # frame cache (uniform 12 frames, raw uint8)
NUM_WORKERS = 8                                    # official default; keep fixed across runs

from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, time
os.makedirs(DRIVE_DIR, exist_ok=True)
# The folder name contains a space, which would split unquoted paths in `!` shell commands -> use a symlink
if os.path.islink(DRIVE_ROOT) and os.readlink(DRIVE_ROOT) != DRIVE_DIR:
    os.remove(DRIVE_ROOT)
if not os.path.exists(DRIVE_ROOT):
    os.symlink(DRIVE_DIR, DRIVE_ROOT)
print("Drive folder:", DRIVE_DIR, "->", DRIVE_ROOT, "| contents:", sorted(os.listdir(DRIVE_ROOT)))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1

EXP = 'xpool_9k_s24'
RUN_DIR = f'{DRIVE_ROOT}/runs/{EXP}'


In [ ]:
# ==== 1. Code: repo + pinned official X-Pool + deps ====
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -r requirements-colab.txt
!bash scripts/setup_xpool.sh
import sys; sys.path.insert(0, f"{REPO}/src")
!PYTHONPATH=src python -m pytest -q tests/ 2>&1 | tail -3


In [ ]:
# ==== Analysis report: per-category, query length, temporal words, hubness, frame-order probes ====
!PYTHONPATH=src python analysis/analyze_dump.py --dump_dir {RUN_DIR}/dump_model_best --data_dir third_party/xpool/data/MSRVTT --out_dir {RUN_DIR}/analysis
print(open(f'{RUN_DIR}/analysis/analysis_report.md').read())


In [ ]:
# ==== Failure-case gallery and attention over frames (needs the frame cache for thumbnails) ====
# Rebuild cache only if missing (CPU runtime: ~slower but fine)
if not os.path.exists(f'{CACHE_DIR}/index.json'):
    !python scripts/prepare_msrvtt.py --drive_dir {DRIVE_ROOT}/data --local_dir {LOCAL_DATA}
    !PYTHONPATH=src python -m tvr.frame_cache build --videos_dir {LOCAL_DATA}/videos --cache_dir {CACHE_DIR}
!PYTHONPATH=src python analysis/failure_gallery.py --dump {RUN_DIR}/dump_model_best/dump_original.pt --ckpt {RUN_DIR}/model_best.pth --cache_dir {CACHE_DIR} --out_dir {RUN_DIR}/analysis/gallery --n 12
